In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset

#Transform: only convert to tensor
transform = transforms.Compose([
    transforms.ToTensor()
])

#Load the dataset
emnist_train = datasets.EMNIST(root='./data', split='byclass', train=True, download=True, transform=transform)

#Get the labels
labels = emnist_train.targets

#Find indices for class 'A' and class 'B'
a_indices = ...
b_indices = ...

print(f"'A' sample count: {len(a_indices)}")
print(f"'B' sample count: {len(b_indices)}")


In [ ]:
def show_samples(dataset, num_samples=6):
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    for i in range(num_samples):
        img, label = dataset[i]
        img = img.squeeze(0)  # 1x28x28 → 28x28
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"Label: {label}")  # .item() removed
        axes[i].axis('off')
    plt.show()

show_samples(ab_dataset)



In [ ]:
X = []
y = []

for img, label in ab_dataset:
    X.append(img.numpy().squeeze().reshape(-1))  # 28x28 -> 784 flat vector
    y.append(label)

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}")  # (number_of_samples, 784)
print(f"y shape: {y.shape}")  # (number_of_samples,)


In [ ]:
# 'A' (10) -> 0, 'B' (11) -> 1
y = np.where(y == 10, 0, 1)

print(np.unique(y))  # will print [0 1]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred_logreg = logreg.predict(X_test)

from sklearn.metrics import classification_report

print("Logistic Regression")
print(classification_report(y_test, y_pred_logreg))


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("KNN")
print(classification_report(y_test, y_pred_knn))

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("SVM")
print(classification_report(y_test, y_pred_svm))


In [ ]:
import joblib

joblib.dump(logreg, 'models/logistic_regression_model.pkl')

joblib.dump(knn, 'models/knn_model.pkl')

joblib.dump(svm, 'models/svm_model.pkl')

In [ ]:
import torch
import torch.nn as nn

class ABClassifierCNN(nn.Module):
    def __init__(self):
        super(ABClassifierCNN, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 2)  
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1) 
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

model = ABClassifierCNN().to(device)

Kullanılan cihaz: cuda


In [ ]:
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:

X_train_tensor = torch.tensor(X_train).view(-1, 1, 28, 28).float()
y_train_tensor = torch.tensor(y_train).long()

X_test_tensor = torch.tensor(X_test).view(-1, 1, 28, 28).float()
y_test_tensor = torch.tensor(y_test).long()

train_loader = torch.utils.data.DataLoader(
    dataset=torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    dataset=torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=64,
    shuffle=False
)

In [ ]:
num_epochs = 5 

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {running_loss/len(train_loader):.4f}")

print("Training completed!")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


In [ ]:
torch.save(model.state_dict(), 'models/ab_classifier_cnn.pt')
print("Model saved as 'models/ab_classifier_cnn.pt'")